# Đánh giá mô hình CNN trên tập dữ liệu CIFAR-10

Notebook này thực hiện quy trình đánh giá toàn diện cho mô hình CNN đã huấn luyện:
1. **Dữ liệu**: Nạp tập kiểm thử `test_batch` và danh sách nhãn lớp từ `cifar-10-batches-py`.
2. **Tiền xử lý**: Xây dựng pipeline `test_ds` (`tf.data.Dataset`) với cùng phép chuẩn hóa kênh RGB (mean & std) như lúc huấn luyện (`build_cnn.ipynb`).
3. **Mô hình**: Nạp checkpoint mô hình đã huấn luyện từ thư mục `checkpoints/` (tự động xử lý tương thích cấu hình).
4. **Đánh giá**:
   - Đo lường `Test Loss` và `Test Accuracy` trực tiếp trên `test_ds`.
   - Lập báo cáo phân loại chi tiết (Classification Report: Precision, Recall, F1-Score).
   - Vẽ và xuất ma trận nhầm lẫn (Confusion Matrix).
   - Trực quan hóa kết quả dự đoán trên các hình ảnh kiểm thử mẫu.
   - Lưu trữ toàn bộ kết quả đánh giá vào thư mục `evaluation/`.

In [ ]:
import csv
import json
import pickle
import tempfile
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix

# Thiết lập đường dẫn thư mục
data_dir_candidates = [
    Path.cwd() / "cifar-10-batches-py",
    Path.cwd().parent / "cifar-10-batches-py",
]
DATA_DIR = next(
    (path.resolve() for path in data_dir_candidates if (path / "data_batch_1").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục cifar-10-batches-py cạnh notebook hoặc project root."
    )

PROJECT_DIR = DATA_DIR.parent
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
EVALUATION_DIR = PROJECT_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

print(f"TensorFlow version: {tf.__version__}")
print(f"Thư mục dữ liệu: {DATA_DIR}")
print(f"Thư mục checkpoints: {CHECKPOINT_DIR}")
print(f"Thư mục lưu kết quả evaluation: {EVALUATION_DIR}")

## 1. Đọc dữ liệu kiểm thử từ CIFAR-10 (`cifar-10-batches-py`)

Đọc trực tiếp file `test_batch` (10.000 ảnh) và `batches.meta` (danh sách tên 10 lớp) từ thư mục `cifar-10-batches-py`.

In [ ]:
def load_cifar_batch(filename):
    """Đọc một batch dữ liệu CIFAR-10 từ file pickle."""
    batch_path = DATA_DIR / filename
    with batch_path.open("rb") as file:
        batch = pickle.load(file, encoding="bytes")

    images = tf.convert_to_tensor(batch[b"data"], dtype=tf.uint8)
    images = tf.reshape(images, [-1, 3, 32, 32])
    images = tf.transpose(images, [0, 2, 3, 1])  # Chuyển từ NCHW sang NHWC
    labels = tf.convert_to_tensor(batch[b"labels"], dtype=tf.int32)
    return images, labels


# Nạp test batch
x_test, y_test = load_cifar_batch("test_batch")

# Nạp nhãn lớp từ batches.meta
with (DATA_DIR / "batches.meta").open("rb") as file:
    metadata = pickle.load(file, encoding="bytes")
class_names = [name.decode("utf-8") for name in metadata[b"label_names"]]

print(f"Tập kiểm thử (Test): {x_test.shape[0]} ảnh")
print(f"Kích thước mỗi ảnh: {x_test.shape[1:]}, dtype={x_test.dtype}")
print(f"Nhãn: {y_test.shape}, dtype={y_test.dtype}")
print(f"Danh sách {len(class_names)} lớp: {class_names}")

## 2. Tiền xử lý và tạo pipeline `test_ds` (`tf.data.Dataset`)

Chuẩn hóa ảnh kiểm thử bằng đúng giá trị RGB Mean và Std đã được tính từ 45.000 ảnh tập huấn luyện trong `build_cnn.ipynb`:
- `PIXEL_MEAN = [0.4914786, 0.48227564, 0.44640514]`
- `PIXEL_STD  = [0.24707623, 0.24351896, 0.2615412]`

Pipeline `test_ds` được xây dựng bằng `tf.data.Dataset` với batching và prefetching để tối ưu hiệu năng tính toán.

In [ ]:
# Giá trị RGB Mean và Std tính từ 45.000 ảnh train (sau khi tách 5.000 validation) trong build_cnn.ipynb
PIXEL_MEAN = tf.constant([0.4914786, 0.48227564, 0.44640514], dtype=tf.float32)
PIXEL_STD = tf.constant([0.24707623, 0.24351896, 0.2615412], dtype=tf.float32)

BATCH_SIZE = 64


def normalize_example(image, label):
    """Chuẩn hóa ảnh về dải [0, 1] rồi z-score normalization theo train mean/std."""
    image = tf.cast(image, tf.float32) / 255.0
    image = (image - PIXEL_MEAN) / PIXEL_STD
    return image, tf.cast(label, tf.int32)


# Xây dựng test_ds bằng tf.data.Dataset
test_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(normalize_example, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Đã tạo test_ds thành công:")
print(f"  - Số lượng mẫu: {len(x_test)}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Cấu trúc phần tử: {test_ds.element_spec}")

## 3. Nạp mô hình đã huấn luyện từ thư mục `checkpoints/`

Nạp mô hình tốt nhất từ `checkpoints/`. Hàm `load_checkpoint` tự động hỗ trợ chuyển đổi tương thích nếu file checkpoint được lưu từ môi trường TF/Keras khác (như TF 2.15 trên Windows với `BatchNormalization` axis list).

In [ ]:
def load_checkpoint(path):
    """
    Nạp model Keras từ checkpoint.
    Tự động xử lý tương thích cấu hình (BatchNormalization axis,
    hoặc đường dẫn HDF5 trên Windows) khi nạp trên Keras 3.
    """
    try:
        return tf.keras.models.load_model(str(path), compile=False)
    except Exception as exc:
        print(f"Nạp trực tiếp gặp lỗi: {exc}")
        print("Đang tự động chuyển đổi định dạng tương thích Keras 3...")
        import h5py

        with tempfile.TemporaryDirectory(prefix="cifar10-compat-") as temp_dir:
            temp_dir = Path(temp_dir)
            old_weights = temp_dir / "old.weights.h5"
            new_weights = temp_dir / "new.weights.h5"
            compatible_checkpoint = temp_dir / "compatible.keras"

            with zipfile.ZipFile(path) as archive:
                config = json.loads(archive.read("config.json"))

                def fix_config(value):
                    if isinstance(value, dict):
                        if value.get("class_name") == "BatchNormalization":
                            axis = value.get("config", {}).get("axis")
                            if isinstance(axis, list) and len(axis) == 1:
                                value["config"]["axis"] = axis[0]
                        for child in value.values():
                            fix_config(child)
                    elif isinstance(value, list):
                        for child in value:
                            fix_config(child)

                fix_config(config)
                old_weights.write_bytes(archive.read("model.weights.h5"))

                with h5py.File(old_weights, "r") as source, h5py.File(new_weights, "w") as target:
                    def copy_item(name, item):
                        target_name = name.replace("\\", "/")
                        if isinstance(item, h5py.Group):
                            group = target.require_group(target_name) if target_name else target
                            for key, value in item.attrs.items():
                                group.attrs[key] = value
                        else:
                            parent, _, leaf = target_name.rpartition("/")
                            group = target.require_group(parent) if parent else target
                            dataset = group.create_dataset(leaf, data=item[()], dtype=item.dtype)
                            for key, value in item.attrs.items():
                                dataset.attrs[key] = value

                    source.visititems(copy_item)

                with zipfile.ZipFile(compatible_checkpoint, "w", zipfile.ZIP_DEFLATED) as output:
                    for name in archive.namelist():
                        if name == "config.json":
                            data = json.dumps(config).encode("utf-8")
                        elif name == "model.weights.h5":
                            data = new_weights.read_bytes()
                        else:
                            data = archive.read(name)
                        output.writestr(name, data)

            return tf.keras.models.load_model(str(compatible_checkpoint), compile=False)


# Chọn checkpoint tốt nhất
preferred_checkpoint = CHECKPOINT_DIR / "cifar10_vgg_style_cnn_best.keras"
if preferred_checkpoint.is_file():
    checkpoint_path = preferred_checkpoint
else:
    # Tìm checkpoint .keras mới nhất trong thư mục
    keras_files = sorted(CHECKPOINT_DIR.glob("*.keras"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not keras_files:
        raise FileNotFoundError(f"Không tìm thấy file .keras nào trong {CHECKPOINT_DIR}")
    checkpoint_path = keras_files[0]

print(f"Đang nạp checkpoint: {checkpoint_path}")
model = load_checkpoint(checkpoint_path)
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)
print("Đã nạp và biên dịch model thành công!")
model.summary()

## 4. Đánh giá mô hình trên tập kiểm thử (`test_ds`)

Chạy `model.evaluate(test_ds)` để tính toán Test Loss và Test Accuracy tổng thể.

In [ ]:
test_eval = model.evaluate(test_ds, return_dict=True, verbose=1)

print("-" * 40)
print(f"Test Loss    : {test_eval['loss']:.4f}")
print(f"Test Accuracy: {test_eval['accuracy'] * 100:.2f}%")
print("-" * 40)

## 5. Dự đoán chi tiết và Báo cáo phân loại (Classification Report)

Tính toán phân phối xác suất dự đoán, trích xuất nhãn dự đoán cho toàn bộ tập kiểm thử, sau đó lập bảng Classification Report với các chỉ số Precision, Recall và F1-Score cho từng lớp. Kết quả được lưu vào `evaluation/classification_report.csv` và `evaluation/test_metrics.json`.

In [ ]:
# Lấy dự đoán phân phối xác suất
y_prob = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_prob, axis=1)
y_true = np.asarray(y_test).reshape(-1)

# Tính classification report
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4,
    output_dict=True,
    zero_division=0,
)

print("\n=== CLASSIFICATION REPORT ===")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0,
    )
)

# Lưu test_metrics.json
metrics = {
    "test_loss": float(test_eval["loss"]),
    "test_accuracy": float(test_eval["accuracy"]),
    "macro_avg": {
        "precision": float(report_dict["macro avg"]["precision"]),
        "recall": float(report_dict["macro avg"]["recall"]),
        "f1_score": float(report_dict["macro avg"]["f1-score"]),
    },
    "weighted_avg": {
        "precision": float(report_dict["weighted avg"]["precision"]),
        "recall": float(report_dict["weighted avg"]["recall"]),
        "f1_score": float(report_dict["weighted avg"]["f1-score"]),
    },
    "per_class": {
        name: {
            "precision": float(report_dict[name]["precision"]),
            "recall": float(report_dict[name]["recall"]),
            "f1_score": float(report_dict[name]["f1-score"]),
            "support": int(report_dict[name]["support"]),
        }
        for name in class_names
    },
}
metrics_file = EVALUATION_DIR / "test_metrics.json"
with metrics_file.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)
    file.write("\n")

# Lưu classification_report.csv
report_columns = ["class", "precision", "recall", "f1_score", "support"]
report_csv_file = EVALUATION_DIR / "classification_report.csv"
with report_csv_file.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=report_columns)
    writer.writeheader()
    for name in class_names:
        writer.writerow({
            "class": name,
            "precision": report_dict[name]["precision"],
            "recall": report_dict[name]["recall"],
            "f1_score": report_dict[name]["f1-score"],
            "support": int(report_dict[name]["support"]),
        })
    for name, key in (("macro_avg", "macro avg"), ("weighted_avg", "weighted avg")):
        writer.writerow({
            "class": name,
            "precision": report_dict[key]["precision"],
            "recall": report_dict[key]["recall"],
            "f1_score": report_dict[key]["f1-score"],
            "support": int(report_dict[key]["support"]),
        })

print(f"Đã lưu kết quả chỉ số vào:\n  - {metrics_file}\n  - {report_csv_file}")

## 6. Ma trận nhầm lẫn (Confusion Matrix)

Tính toán ma trận nhầm lẫn để phân tích các trường hợp mô hình nhầm lẫn giữa các lớp (ví dụ nhầm giữa cat và dog, hoặc giữa automobile và truck). Ma trận được vẽ biểu đồ nhiệt và lưu vào `evaluation/confusion_matrix.png` cùng bảng `evaluation/confusion_matrix.csv`.

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# Lưu ma trận nhầm lẫn ra CSV
cm_csv_file = EVALUATION_DIR / "confusion_matrix.csv"
with cm_csv_file.open("w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["true/predicted", *class_names])
    for name, row in zip(class_names, cm):
        writer.writerow([name, *row.tolist()])

# Vẽ Confusion Matrix
fig, ax = plt.subplots(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    colorbar=False,
    xticks_rotation=45,
)
ax.set_title("CIFAR-10 Test Confusion Matrix", fontsize=14, fontweight="bold", pad=12)
plt.tight_layout()

cm_png_file = EVALUATION_DIR / "confusion_matrix.png"
fig.savefig(cm_png_file, dpi=200, bbox_inches="tight")
plt.show()

print(f"Đã lưu ma trận nhầm lẫn vào:\n  - {cm_csv_file}\n  - {cm_png_file}")

## 7. Trực quan hóa một số hình ảnh dự đoán mẫu

Hiển thị ngẫu nhiên 16 ảnh từ tập kiểm thử cùng nhãn thực tế, nhãn dự đoán và độ tin cậy (Confidence).
- Màu xanh lá: Dự đoán chính xác.
- Màu đỏ: Dự đoán sai.

In [ ]:
np.random.seed(42)
sample_indices = np.random.choice(len(x_test), size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for idx, ax in zip(sample_indices, axes.flat):
    img = x_test[idx].numpy() if hasattr(x_test[idx], "numpy") else x_test[idx]
    true_label = class_names[y_true[idx]]
    pred_label = class_names[y_pred[idx]]
    confidence = y_prob[idx][y_pred[idx]] * 100.0

    is_correct = (y_true[idx] == y_pred[idx])
    color = "#2e7d32" if is_correct else "#c62828"

    ax.imshow(img)
    ax.set_title(
        f"Thực: {true_label}\nĐoán: {pred_label} ({confidence:.1f}%)",
        color=color,
        fontsize=10,
        fontweight="bold",
    )
    ax.axis("off")

plt.suptitle("Một số dự đoán mẫu trên tập kiểm thử (Xanh: Đúng, Đỏ: Sai)", fontsize=15, fontweight="bold", y=0.99)
plt.tight_layout()

sample_preds_file = EVALUATION_DIR / "sample_predictions.png"
fig.savefig(sample_preds_file, dpi=200, bbox_inches="tight")
plt.show()

print(f"Đã lưu hình ảnh dự đoán mẫu vào: {sample_preds_file}")